# RAG Agent - ChromaDB

Day14 arabalar: excel -> chroma -> soru. TF-IDF cosine degil.


In [ ]:
import pandas as pd
import chromadb


### Data


In [ ]:
df=pd.read_excel('data/cars.xls')
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


### Bos veri


In [ ]:
df=df.dropna()


### Vektor DB (ders)


In [ ]:
col=chromadb.PersistentClient(path='./cars_db').get_or_create_collection('car_data')
docs=df.apply(lambda r: f'{r.Price},{r.Mileage},{r.Make} {r.Model} {r.Trim} {r.Type} {r.Cylinder} cylinders',axis=1).tolist()
col.add(
    ids=[str(i) for i in df.index],
    documents=docs,
    metadatas=df.to_dict('records')
)
print('Indexed',len(df),'cars')


### Retrieval


In [ ]:
col=chromadb.PersistentClient(path='./cars_db').get_collection('car_data')
res=col.query(query_texts=['4 cylinder sedan with high mileage'],n_results=3)
print(res['documents'][0])


### RAG agent


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from IPython.display import Markdown

client=OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv('OPENROUTER_API_KEY'),
)

def chat_with_cars(query):
    context='\n'.join(col.query(query_texts=[query],n_results=3)['documents'][0])
    prompt=f'Car Salesman. Use ONLY context. Format: ## Heading, **Bold**, Bullets.\n\nContext: {context}\n\nUser: {query}'
    response=client.chat.completions.create(
        model='openai/gpt-oss-120b:free',
        messages=[{'role':'user','content':prompt}]
    )
    return Markdown(response.choices[0].message.content)


In [ ]:
chat_with_cars('What is the cheapest 4 cylinder sedan with high mileage?')


### ollama ile ayni retrieve


In [ ]:
import ollama
ctx='\n'.join(col.query(query_texts=['reliable 4 cylinder car for a student'],n_results=3)['documents'][0])
r=ollama.chat(model='llama3.2-vision',messages=[{
    'role':'user',
    'content':f'Car salesman. ONLY this context:\n{ctx}\n\nQ: student icin 4 silindir oner'
}])
print(r['message']['content'])


### Sonuc

Chroma getiriyor, LLM cevap yaziyor. Bu RAG agent. TF-IDF argmax degil.
